### Import Library & Load Dataset

In [56]:
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

df_raw = pd.read_csv('../dataset/jakarta_house_v3.csv')
print(f"Dataset awal berhasil dimuat! Ukuran data: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom")
display(df_raw.head())

Dataset awal berhasil dimuat! Ukuran data: 9533 baris, 10 kolom


,index,price,district,city,bed_rooms,bath_rooms,carport,land_area,building_area,njop/m2
0,1,5900000000,Kalideres,Jakarta Barat,2,4,2,250.0,350.0,3843000.0
1,2,2700000000,Grogol Petamburan,Jakarta Barat,4,2,0,100.0,225.0,11305000.0
2,3,2200000000,Grogol Petamburan,Jakarta Barat,3,3,0,60.0,140.0,11305000.0
3,4,1900000000,Grogol Petamburan,Jakarta Barat,3,2,0,60.0,120.0,11305000.0
4,5,2100000000,Grogol Petamburan,Jakarta Barat,4,3,0,56.0,108.0,11305000.0


### Menghapus Kolom Yang Tidak Perlu

In [57]:
df = df_raw.drop(columns=['index']).copy()

print(f"Hasil Column Filter ({df.shape[1]} kolom):")
display(df.head())

Hasil Column Filter (9 kolom):


,price,district,city,bed_rooms,bath_rooms,carport,land_area,building_area,njop/m2
0,5900000000,Kalideres,Jakarta Barat,2,4,2,250.0,350.0,3843000.0
1,2700000000,Grogol Petamburan,Jakarta Barat,4,2,0,100.0,225.0,11305000.0
2,2200000000,Grogol Petamburan,Jakarta Barat,3,3,0,60.0,140.0,11305000.0
3,1900000000,Grogol Petamburan,Jakarta Barat,3,2,0,60.0,120.0,11305000.0
4,2100000000,Grogol Petamburan,Jakarta Barat,4,3,0,56.0,108.0,11305000.0


### Penanganan Missing Value

In [58]:
print("Jumlah missing value sebelum ditangani:")
print(df.isnull().sum())

# Mengisi missing value pada land_area dan building_area menggunakan nilai median
df['land_area'] = df['land_area'].fillna(df['land_area'].median())
df['building_area'] = df['building_area'].fillna(df['building_area'].median())

print("\nJumlah missing value setelah ditangani:")
print(df.isnull().sum())
print(f"\nUkuran data setelah penanganan missing value: {df.shape[0]} baris, {df.shape[1]} kolom")


Jumlah missing value sebelum ditangani:
price            0
district         0
city             0
bed_rooms        0
bath_rooms       0
carport          0
land_area        3
building_area    5
njop/m2          0
dtype: int64

Jumlah missing value setelah ditangani:
price            0
district         0
city             0
bed_rooms        0
bath_rooms       0
carport          0
land_area        0
building_area    0
njop/m2          0
dtype: int64

Ukuran data setelah penanganan missing value: 9533 baris, 9 kolom


### Pembersihan Data Duplikat

In [59]:
print("Jumlah data duplikat:", df.duplicated().sum())

df_cleaned = df.drop_duplicates().reset_index(drop=True)

print(f"\nUkuran data setelah hapus duplikat: {df_cleaned.shape[0]} baris")
print("Jumlah data duplikat setelah penghapusan :", df_cleaned.duplicated().sum())

Jumlah data duplikat: 646

Ukuran data setelah hapus duplikat: 8887 baris
Jumlah data duplikat setelah penghapusan : 0


### Deteksi & Penanganan Outlier

In [60]:
# kolom_numerik = [
#     'price', 
#     'bed_rooms', 
#     'bath_rooms', 
#     'carport', 
#     'land_area', 
#     'building_area',
#     'njop/m2',
# ]

# # Menghitung Q1, Q3, dan IQR (1.5) pada kolom numerik
# Q1 = df_cleaned[kolom_numerik].quantile(0.25)
# Q3 = df_cleaned[kolom_numerik].quantile(0.75)
# IQR = Q3 - Q1

# batas_bawah = Q1 - 1.5 * IQR
# batas_atas  = Q3 + 1.5 * IQR

# # Mendeteksi outlier sebelum ditangani
# kondisi_outlier = (df_cleaned[kolom_numerik] < batas_bawah) | (df_cleaned[kolom_numerik] > batas_atas)

# tabel_sebelum = pd.DataFrame({
#     'Jumlah Data': len(df_cleaned),
#     'Jumlah Outlier': kondisi_outlier.sum(),
#     'Batas Bawah': batas_bawah.round(2),
#     'Batas Atas': batas_atas.round(2)
# })
# print("TABEL ANALISIS JUMLAH OUTLIER SEBELUM DITANGANI:")
# display(tabel_sebelum)

# # Menghapus baris yang memiliki nilai outlier
# df_bersih_outlier = df_cleaned[~kondisi_outlier.any(axis=1)].reset_index(drop=True)

# # Memeriksa kembali outlier pada batas yang sama setelah pembersihan
# kondisi_outlier_setelah = (df_bersih_outlier[kolom_numerik] < batas_bawah) | (df_bersih_outlier[kolom_numerik] > batas_atas)

# tabel_sesudah = pd.DataFrame({
#     'Jumlah Data': len(df_bersih_outlier),
#     'Jumlah Outlier': kondisi_outlier_setelah.sum(),
#     'Batas Bawah': batas_bawah.round(2),
#     'Batas Atas': batas_atas.round(2)
# })
# print("\nTABEL HASIL SETELAH PENANGANAN OUTLIER:")
# display(tabel_sesudah)

# print(f"\nTotal data sebelum penanganan outlier : {len(df_cleaned)} baris")
# print(f"Total data setelah penanganan outlier : {len(df_bersih_outlier)} baris")

### Feature Engineering

In [61]:
df_fe = df_cleaned.copy()

# Total NJOP Tanah (Luas Tanah x NJOP per m2)
df_fe['total_njop_land'] = df_fe['land_area'] * df_fe['njop/m2']

### Feature Selection

In [62]:
fitur_digunakan = [
    'price',
    'district',
    'city',
    'bed_rooms',
    'bath_rooms',
    'carport',
    'land_area',
    'building_area',
    'njop/m2',
    'total_njop_land',
]

df_selected = df_fe[fitur_digunakan].copy()

print(f"Hasil Feature Selection ({df_selected.shape[1]} kolom dipilih):")
display(df_selected.head())

Hasil Feature Selection (10 kolom dipilih):


,price,district,city,bed_rooms,bath_rooms,carport,land_area,building_area,njop/m2,total_njop_land
0,5900000000,Kalideres,Jakarta Barat,2,4,2,250.0,350.0,3843000.0,9.607500e+08
1,2700000000,Grogol Petamburan,Jakarta Barat,4,2,0,100.0,225.0,11305000.0,1.130500e+09
2,2200000000,Grogol Petamburan,Jakarta Barat,3,3,0,60.0,140.0,11305000.0,6.783000e+08
3,1900000000,Grogol Petamburan,Jakarta Barat,3,2,0,60.0,120.0,11305000.0,6.783000e+08
4,2100000000,Grogol Petamburan,Jakarta Barat,4,3,0,56.0,108.0,11305000.0,6.330800e+08


### Transformasi Logaritmik Target

In [63]:
X = df_selected.drop(columns=['price'])
y = np.log1p(df_selected['price'])

print("Perbandingan target sebelum dan sesudah transformasi logaritmik:")
display(pd.DataFrame({
    'Price Asli (Rp)': df_selected['price'].head(),
    'Price Log': y.head()
}))

Perbandingan target sebelum dan sesudah transformasi logaritmik:


,Price Asli (Rp),Price Log
0,5900000000,22.498218
1,2700000000,21.716518
2,2200000000,21.511723
3,1900000000,21.365120
4,2100000000,21.465203


### One-Hot Encoding Menggunakan Column Transformer

In [64]:
kolom_kategorikal = ['district', 'city']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), kolom_kategorikal)
    ],
    remainder='passthrough'
)

X_encoded_preview = preprocessor.fit_transform(X)

print(f"Jumlah fitur sebelum One-Hot Encoding: {X.shape[1]} fitur")
print(f"Jumlah fitur setelah One-Hot Encoding: {X_encoded_preview.shape[1]} fitur")

Jumlah fitur sebelum One-Hot Encoding: 9 fitur
Jumlah fitur setelah One-Hot Encoding: 54 fitur


### Partitioning Data (80% Training : 20% Testing)

In [65]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, shuffle=True
)

print(f"Jumlah Data Training (80%): {len(X_train)} baris")
print(f"Jumlah Data Testing  (20%): {len(X_test)} baris")

Jumlah Data Training (80%): 7109 baris
Jumlah Data Testing  (20%): 1778 baris


### Menyimpan Data Hasil Preprocessing

In [66]:
data_lengkap = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'preprocessor': preprocessor,
    'df_bersih': df_selected
}

joblib.dump(data_lengkap, '../dataset/data_hasil_preprocessing.pkl')
print("Data siap pemodelan berhasil disimpan ke '../dataset/data_hasil_preprocessing.pkl'!")

Data siap pemodelan berhasil disimpan ke '../dataset/data_hasil_preprocessing.pkl'!
